In [2]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [3]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [4]:
len(words) #dataset size

32033

In [5]:
chars = sorted(list(set(''.join(words)))) #build the vocabulary of characters
stoi = {s:i+1 for i,s in enumerate(chars)} #letter to integer(index) mapping    
stoi['.'] = 0 #index 0 will be reserved for the end of a word and start of the word
itos = {i:s for s,i in stoi.items()} # reverse mapping from integer(index) to letter
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [6]:
block_size = 3 # context length: how many characters do we take to predict the next one? #upgrade to bigram level
X, Y = [], [] #inputs and targets split variables
for w in words:
  
  #print(w)
  context = [0] * block_size 
  for ch in w + '.':
    ix = stoi[ch]
    X.append(context)
    Y.append(ix)
    #print(''.join(itos[i] for i in context), '--->', itos[ix])
    context = context[1:] + [ix] # crop and append
  
X = torch.tensor(X)
Y = torch.tensor(Y)

In [7]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([228146, 3]), torch.int64, torch.Size([228146]), torch.int64)

In [8]:
# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one?

def build_dataset(words):  
  X, Y = [], []
  for w in words:

    #print(w)
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      #print(''.join(itos[i] for i in context), '--->', itos[ix])
      context = context[1:] + [ix] # crop and append

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [9]:
C = torch.randn((27, 2))

In [10]:
emb = C[X]
emb.shape

torch.Size([228146, 3, 2])

In [11]:
W1 = torch.randn((6, 100))
b1 = torch.randn(100)

In [12]:
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)

In [13]:
h

tensor([[ 0.1220, -0.7818, -0.5023,  ..., -0.1565,  0.7198, -0.8336],
        [-0.7188, -0.2990, -0.9617,  ..., -0.1777,  0.5398, -0.5698],
        [ 0.1618, -0.9747,  0.2740,  ..., -0.5660,  0.5573, -0.8213],
        ...,
        [-0.7242, -0.3323,  0.9870,  ..., -0.3703, -0.8126, -0.7480],
        [ 0.2462,  0.7004,  0.9254,  ...,  0.2478,  0.0887, -0.6440],
        [-0.9964, -0.9317, -0.7822,  ..., -0.0799, -0.8036, -0.8598]])

In [14]:
h.shape

torch.Size([228146, 100])

In [15]:
W2 = torch.randn((100, 27))
b2 = torch.randn(27)

In [16]:
logits = h @ W2 + b2

In [17]:
logits.shape

torch.Size([228146, 27])

In [18]:
counts = logits.exp()

In [19]:
prob = counts / counts.sum(1, keepdims=True)

In [20]:
prob.shape

torch.Size([228146, 27])

In [21]:
Y   #the actual target values

tensor([ 5, 13, 13,  ..., 26, 24,  0])

In [22]:
loss = -prob[torch.arange(228146), Y].log().mean() #static code to calculate the loss
loss

tensor(17.8512)

In [23]:

Xtr.shape, Ytr.shape # dataset

(torch.Size([182625, 3]), torch.Size([182625]))

In [24]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27, 2), generator=g) #embeddings matrix   
W1 = torch.randn((6, 100), generator=g) #hidden layer weights
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g) #second hidden layer weights
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2] 

In [25]:
sum(p.nelement() for p in parameters) # number of parameters in total

3481

In [26]:
for p in parameters:
  p.requires_grad = True

In [27]:
lre = torch.linspace(-3, 0, 1000)
lrs = 10**lre

In [ ]:
lri = [] #track the learning rate for each iteration
lossi = [] #track the loss for each iteration

In [ ]:
for i in range(1000):

  # do it in minibatches
  ix = torch.randint(0, Xtr.shape[0], (32,))
  
  # forward pass
  emb = C[Xtr[ix]] # (32, 3, 10)
  h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # (32, 100)
  logits = h @ W2 + b2 # (32, 27)
  loss = F.cross_entropy(logits, Ytr[ix])
  print(loss.item())
  
  # backward pass
  for p in parameters:
    p.grad = None
  loss.backward()
  
  # update
  lr = lrs[i] #learning rate
  for p in parameters:
    p.data += -lr * p.grad

#print(loss.item())

2.456456184387207
2.5585389137268066
2.860964059829712
2.3952605724334717
2.472442150115967
2.7255589962005615
2.36924147605896
3.004694938659668
2.816650152206421
2.1874420642852783
2.598087787628174
2.8349947929382324
2.39785099029541
2.6092472076416016
2.811830520629883
2.348318099975586
2.3906803131103516
2.666283130645752
2.593672752380371
2.439668893814087
2.7661731243133545
2.424445152282715
2.6193015575408936
2.6257381439208984
2.880174160003662
2.561526298522949
2.6271328926086426
3.0511937141418457
2.387028455734253
3.0956943035125732
3.187431812286377
3.302048921585083
2.423497200012207
2.9753923416137695
3.0751631259918213
2.8208634853363037
2.7858235836029053
2.7458178997039795
2.941776752471924
2.684730052947998
2.5391688346862793
2.831073045730591
3.055058717727661
2.6500203609466553
2.770904064178467
2.714996814727783
2.5355842113494873
2.614058017730713
2.891157627105713
2.8588874340057373
3.018150568008423
2.7077696323394775
2.967857599258423
2.8949928283691406
2.5335